In [1]:
import numpy as np
import pandas as pd

In [2]:
spec = pd.read_csv("../2일차_실습//02-01_측정의_3요소_설비태그목록.csv")

In [3]:
sample = pd.read_csv(
    "../2일차_실습//02-01_측정의_3요소_측정샘플.csv", parse_dates=["timestamp"]
)

In [4]:
gaps = sample["timestamp"].diff().dt.total_seconds().dropna()
print("=== 실측 timestamp 간격 ===")
print(gaps.value_counts().sort_index().rename_axis("초").to_string())

summary = []
for tag in sample.columns[1:]:
    values = sample[tag].dropna()
    changes = values.diff().abs()
    nonzero = changes[changes > 1e-9]
    row = spec.loc[spec["tag"] == tag]
    if row.empty:
        raise ValueError(f"태그 사양을 찾을 수 없습니다: {tag}")
    info = row.iloc[0]
    summary.append(
        {
            "tag": tag,
            "단위": info["unit"],
            "사양주기_sec": int(info["sampling_sec"]),
            "실제주기_sec": int(gaps.mode().iloc[0]),
            "관측최소": values.min(),
            "관측최대": values.max(),
            "최소비영변화": nonzero.min() if not nonzero.empty else np.nan,
            "사양분해능": info["resolution"],
            "상한반복": int((values == info["range_max"]).sum()),
            "직접/파생": "파생" if "계산값" in str(info["description"]) else "직접",
            "설치위치": info["install_location"],
        }
    )
result = pd.DataFrame(summary)
print("\n=== 측정 정보 요약 ===")
print(result.to_string(index=False))
print("\n=== 3요소 설명 예시 ===")
print(
    "MTR01_VIB_RMS_H: 1번 모터 구동측 베어링 하우징의 수평 진동 실효값을 "
    "mm/s 단위로, 사양상 60초 간격으로 측정합니다."
)
print(
    "HYD01_PRESS_IN은 사양상 매분 직접 측정, FUR01_TEMP_Z1은 5분 사양이지만 "
    "샘플 파일에는 1분 행으로 기록됩니다. 5분 주기의 반복값은 원래 주기인지 "
    "상위 시스템의 보간·전방 채움인지 확인이 필요합니다."
)
print(
    "HYD01_PRESS_IN은 관측 상한 210 bar가 사양 상한과 일치하고 반복됩니다. "
    "포화 가능성이 있으나 실제 압력이 상한에 고정되었는지, 센서·변환·저장 과정의 "
    "클리핑인지 확인 전에는 확정할 수 없습니다."
)


=== 실측 timestamp 간격 ===
초
60.0    119

=== 측정 정보 요약 ===
            tag   단위  사양주기_sec  실제주기_sec   관측최소   관측최대  최소비영변화  사양분해능  상한반복 직접/파생            설치위치
MTR01_VIB_RMS_H mm/s        60        60    1.7    2.4     0.1    0.1     0    직접  구동측 베어링 하우징 수평
  MTR01_CURRENT    A        60        60   62.0   66.0     0.5    0.5     0    직접   모터 제어반 전류 변성기
     MTR01_TEMP degC        60        60   48.0   54.0     0.5    0.5     0    직접 고정자 권선 매입 측온저항체
 HYD01_PRESS_IN  bar        60        60  151.0  210.0     1.0    1.0    14    직접        펌프 토출 배관
  FUR01_TEMP_Z1 degC       300        60 1180.0 1191.5     0.5    0.5     0    직접       1존 천장 열전대

=== 3요소 설명 예시 ===
MTR01_VIB_RMS_H: 1번 모터 구동측 베어링 하우징의 수평 진동 실효값을 mm/s 단위로, 사양상 60초 간격으로 측정합니다.
HYD01_PRESS_IN은 사양상 매분 직접 측정, FUR01_TEMP_Z1은 5분 사양이지만 샘플 파일에는 1분 행으로 기록됩니다. 5분 주기의 반복값은 원래 주기인지 상위 시스템의 보간·전방 채움인지 확인이 필요합니다.
HYD01_PRESS_IN은 관측 상한 210 bar가 사양 상한과 일치하고 반복됩니다. 포화 가능성이 있으나 실제 압력이 상한에 고정되었는지, 센서·변환·저장 과정의 클리핑인지 확인 전에는 확정할 수 없습니다.
